In [4]:
import zipfile, os

DRIVE = '/content/drive/MyDrive'

zip_path = f'{DRIVE}/VisDrone2019-DET-test-challenge.zip'
out_path = f'{DRIVE}/VisDrone2019-DET-test-challenge'

if os.path.exists(zip_path):
    print('Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DRIVE)
    print('Done!')
else:
    print('Zip not found — checking Drive root...')
    for f in os.listdir(DRIVE):
        if 'challenge' in f.lower() or 'test' in f.lower():
            print(f)

Extracting...
Done!


In [5]:
!pip install ultralytics -q
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import os
from pathlib import Path

DRIVE = '/content/drive/MyDrive'
RESULTS_DIR = f'{DRIVE}/yolo_results'

# Check weights exist
weights_path = f'{RESULTS_DIR}/exp_yolo11m_final/weights/best.pt'
print(f'Weights exist: {os.path.exists(weights_path)}')

# Load model
model = YOLO(weights_path)

# Test challenge images
TEST_DIR = f'{DRIVE}/VisDrone2019-DET-test-challenge/images'
SUBMIT_DIR = f'{RESULTS_DIR}/test_challenge_predictions'
os.makedirs(SUBMIT_DIR, exist_ok=True)

print(f'Test images exist: {os.path.exists(TEST_DIR)}')
print(f'Running inference...')

results = model.predict(
    source=TEST_DIR,
    imgsz=640,
    conf=0.25,
    save=False,
    verbose=False
)

for r in results:
    img_name = Path(r.path).stem
    lines = []
    if r.boxes is not None and len(r.boxes):
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            w, h = x2 - x1, y2 - y1
            score = float(box.conf[0])
            cls = int(box.cls[0])
            visdrone_cat = cls + 1
            lines.append(f'{int(x1)},{int(y1)},{int(w)},{int(h)},{score:.4f},{visdrone_cat},0,0')
    with open(f'{SUBMIT_DIR}/{img_name}.txt', 'w') as f:
        f.write('\n'.join(lines))

print(f'Done! {len(results)} images processed')
print(f'Saved to: {SUBMIT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Weights exist: True
Test images exist: True
Running inference...
WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

Done! 1580 images processed
Saved to: /content/drive/MyDrive/yolo_results/test_challenge_predictions
